# 11.12 — DQN Variants

DQN variants keep the same core promise — learn action values from delayed reward — but each variant repairs a different failure mode in bootstrapped value learning. In this lesson, we build the target, Double DQN, dueling values, prioritized replay, and a tiny Rainbow-style combination from scratch with NumPy so the moving pieces stay visible.

## 📖 Concept walkthrough — build each idea from scratch

Before the terse worked examples, we build DQN variants one idea at a time. Run each cell in order and read the printed intermediate values — every target, selection rule, and sampling probability is spelled out so the variants do not feel like names pasted onto a neural network. This walkthrough is self-contained and uses a `_w` suffix on variables so it never clashes with the examples below.

In [ ]:
import numpy as np  # arrays, vectorized Bellman targets, and replay sampling.
import matplotlib.pyplot as plt  # compact plots for targets, values, and priorities.
np.random.seed(0)  # reproducibility for all stochastic demos.

### 1. Bootstrapped DQN target: reward now plus discounted consequence

A DQN estimates one number per action: $Q(s,a)$, the discounted consequence of taking action $a$ in state $s$ and behaving well afterward. The one-step target is $y=r+\gamma\max_{a'}Q_{target}(s',a')$. The reward $r$ is observed, but the future term is bootstrapped from the model's own estimate, so this target is powerful and dangerous: it learns before a full return is known, but errors can feed back into themselves.

In [ ]:
rewards_w = np.array([1.0, 0.0, 2.0])  # three rewards from a short trajectory.
gamma_w = 0.9  # discount: one step later is worth 90% as much as now.
powers_w = gamma_w ** np.arange(len(rewards_w))  # [1, gamma, gamma^2].
return_w = float(np.sum(powers_w * rewards_w))  # discounted return G.
print("discount powers:", np.round(powers_w, 3))
print("three-step return:", round(return_w, 3))
assert round(return_w, 3) == 2.620

▶ What you'll see: the delayed reward of 2 counts as 1.62 because it arrives two discounted steps later.

In [ ]:
q_next_target_w = np.array([0.8, 0.3, 0.6])  # target-network estimates for the next state.
r_w = 1.0  # immediate reward from the sampled transition.
q_old_w = 0.4  # current estimate for Q(s,a) before the update.
alpha_w = 0.5  # table-style learning rate for a visible update.
y_dqn_w = r_w + gamma_w * np.max(q_next_target_w)  # DQN target.
q_new_w = q_old_w + alpha_w * (y_dqn_w - q_old_w)  # move partway toward target.
print("target y:", round(y_dqn_w, 3))
print("updated Q:", round(q_new_w, 3))
assert round(y_dqn_w, 3) == 1.720 and round(q_new_w, 3) == 1.060

▶ What you'll see: the update moves halfway from 0.4 toward the bootstrapped target 1.72.

In [ ]:
plt.figure(figsize=(4.4, 3))
plt.bar(["old Q", "target y", "new Q"], [q_old_w, y_dqn_w, q_new_w], color=["gray", "black", "seagreen"])
plt.title("1: bootstrapped target moves the estimate")
plt.ylabel("value")
plt.show()

▶ What you'll see: the new value sits between the old estimate and the target, not all the way at either endpoint.

*Why it's done this way:* the Bellman equation says good action values equal immediate reward plus discounted future value. DQN replaces the unknown future with a target-network estimate, and the partial update limits how much one noisy transition can rewrite the value table.

### 2. Double DQN: separate action selection from action evaluation

Plain DQN uses the same next-state values to choose the max action and evaluate it. If estimates are noisy, the max operator tends to pick overestimated actions. Double DQN splits the job: the online network chooses $a^*=\arg\max_a Q_{online}(s',a)$, while the target network evaluates that chosen action with $Q_{target}(s',a^*)$.

In [ ]:
q_online_next_w = np.array([1.20, 1.10, 0.70])  # online network chooses the greedy next action.
q_target_next_w = np.array([0.55, 0.95, 0.65])  # target network evaluates the chosen action.
a_star_w = int(np.argmax(q_online_next_w))  # action selected by online network.
print("online next Q:", q_online_next_w)
print("target next Q:", q_target_next_w)
print("online-selected action:", a_star_w)
assert a_star_w == 0

▶ What you'll see: the online network chooses action 0 because 1.20 is its largest estimate.

In [ ]:
plain_target_w = r_w + gamma_w * np.max(q_target_next_w)  # target-net max evaluates its own best action.
double_target_w = r_w + gamma_w * q_target_next_w[a_star_w]  # target-net evaluates online's chosen action.
print("plain DQN target:", round(plain_target_w, 3))
print("Double DQN target:", round(double_target_w, 3))
assert round(plain_target_w, 3) == 1.855 and round(double_target_w, 3) == 1.495

▶ What you'll see: Double DQN is lower here because target-network evaluation does not accept target action 1's larger value unless online also selected it.

In [ ]:
x_w = np.arange(3)
plt.figure(figsize=(5, 3))
plt.bar(x_w - 0.18, q_online_next_w, width=0.36, label="online chooses", color="steelblue")
plt.bar(x_w + 0.18, q_target_next_w, width=0.36, label="target evaluates", color="orange")
plt.axvline(a_star_w, color="black", linestyle="--", label="a*")
plt.xticks(x_w, ["a0", "a1", "a2"])
plt.title("2: Double DQN decouples choose vs evaluate")
plt.legend()
plt.show()

▶ What you'll see: the dashed line marks the online-chosen action, even though the target network's largest bar is elsewhere.

*Why it's done this way:* maximization over noisy estimates has positive bias because the largest noisy sample is likely too large. Double DQN keeps greedy control from the online network but asks a separate target network for the value, reducing the winner's-curse effect in the target.

### 3. Dueling DQN: value plus advantage with an identifiability fix

A dueling network predicts a state value $V(s)$ and action advantages $A(s,a)$, then combines them into $Q(s,a)$. The trick is that $V+A$ is not identifiable: adding 10 to $V$ and subtracting 10 from every advantage gives the same $Q$. Dueling fixes this by subtracting the mean advantage: $Q(s,a)=V(s)+A(s,a)-\frac{1}{|A|}\sum_b A(s,b)$.

In [ ]:
V_w = 2.0  # state value: how good the state is before caring which action.
A_raw_w = np.array([0.5, -0.2, 0.1])  # raw action-specific advantages.
A_centered_w = A_raw_w - np.mean(A_raw_w)  # identifiability fix: mean advantage becomes 0.
Q_duel_w = V_w + A_centered_w  # combine state value and centered advantages.
print("raw advantages:", A_raw_w)
print("centered advantages:", np.round(A_centered_w, 3))
print("dueling Q:", np.round(Q_duel_w, 3))
assert round(float(np.mean(A_centered_w)), 10) == 0.0

▶ What you'll see: the centered advantages sum to zero, so the average Q equals the state value.

In [ ]:
print("mean Q:", round(float(np.mean(Q_duel_w)), 3), "state V:", V_w)
print("greedy action:", int(np.argmax(Q_duel_w)))
assert round(float(np.mean(Q_duel_w)), 3) == 2.000 and int(np.argmax(Q_duel_w)) == 0

▶ What you'll see: action 0 wins because it has the highest advantage, while the whole state still averages to value 2.

In [ ]:
plt.figure(figsize=(5, 3))
plt.bar(["V"] + [f"A{i}-mean(A)" for i in range(3)], [V_w] + list(A_centered_w), color=["gray", "teal", "teal", "teal"])
plt.axhline(0, color="black", linewidth=0.8)
plt.title("3: dueling separates state value from action gaps")
plt.ylabel("component")
plt.show()

▶ What you'll see: a shared state baseline plus positive/negative action adjustments.

*Why it's done this way:* in many states, choosing the exact action barely matters, so learning a shared state value is easier than relearning that baseline for every action. Subtracting mean advantage makes the decomposition unique enough for optimization because only relative action gaps remain in $A$.

### 4. Prioritized replay: sample surprising transitions more often

Uniform replay treats every stored transition equally. Prioritized replay samples transitions with large temporal-difference errors more often, because those transitions currently disagree most with the value estimate. A common rule is $p_i\propto (|\delta_i|+\epsilon)^\alpha$, with importance weights to soften the bias introduced by non-uniform sampling.

In [ ]:
td_errors_w = np.array([0.05, 0.20, 1.00, 0.40, 2.00])  # absolute surprise candidates.
eps_w = 1e-3  # small positive value so no item has exactly zero priority.
alpha_per_w = 0.6  # prioritization strength: 0 is uniform, 1 is fully proportional.
priorities_w = (np.abs(td_errors_w) + eps_w) ** alpha_per_w
prob_w = priorities_w / np.sum(priorities_w)
print("priorities:", np.round(priorities_w, 3))
print("sampling probabilities:", np.round(prob_w, 3))
assert int(np.argmax(prob_w)) == 4

▶ What you'll see: the transition with TD error 2.0 receives the largest sampling probability.

In [ ]:
N_w = len(td_errors_w)
beta_w = 0.4  # correction strength; rises toward 1 later in training.
weights_w = (N_w * prob_w) ** (-beta_w)  # inverse-probability correction.
weights_w = weights_w / np.max(weights_w)  # normalize so the largest weight is 1.
print("importance weights:", np.round(weights_w, 3))
assert round(float(np.max(weights_w)), 3) == 1.000

▶ What you'll see: frequently sampled high-priority items get smaller correction weights, while rare items get larger ones.

In [ ]:
plt.figure(figsize=(5, 3))
plt.bar(np.arange(N_w) - 0.18, prob_w, width=0.36, label="sample prob", color="purple")
plt.bar(np.arange(N_w) + 0.18, weights_w, width=0.36, label="IS weight", color="orange")
plt.title("4: PER samples surprise but corrects bias")
plt.xlabel("transition id")
plt.legend()
plt.show()

▶ What you'll see: high-error transitions are sampled more, but their learning weights are partly reduced.

*Why it's done this way:* TD error is a practical proxy for learning potential: a transition whose target and prediction disagree can change the model. Non-uniform sampling changes the data distribution, so importance weights push the update back toward the uniform-replay objective.

### 5. Rainbow-style combination: independent fixes stack into one target

Rainbow is not one new Bellman equation; it combines several useful DQN repairs. In a tiny NumPy version, we can combine Double DQN's target choice, dueling's $Q=V+A-\bar A$ construction, and prioritized replay's weighted loss. The point is to see that each variant touches a different part of the learning pipeline.

In [ ]:
V_online_w = np.array([1.1])  # online value stream for the next state.
A_online_w = np.array([[0.2, 0.4, -0.1]])  # online advantages choose the action.
V_target_w = np.array([0.9])  # target value stream for the same next state.
A_target_w = np.array([[0.1, 0.0, 0.3]])  # target advantages evaluate that action.
Q_online_w = V_online_w[:, None] + A_online_w - A_online_w.mean(axis=1, keepdims=True)
Q_target_w = V_target_w[:, None] + A_target_w - A_target_w.mean(axis=1, keepdims=True)
print("online dueling Q:", np.round(Q_online_w, 3))
print("target dueling Q:", np.round(Q_target_w, 3))

▶ What you'll see: both networks produce action values by combining a shared state value with centered advantages.

In [ ]:
a_next_w = int(np.argmax(Q_online_w[0]))
rainbow_y_w = 1.0 + gamma_w * Q_target_w[0, a_next_w]
q_sa_w = 0.6
td_rainbow_w = rainbow_y_w - q_sa_w
weighted_loss_w = weights_w[2] * (td_rainbow_w ** 2)  # pretend transition id 2 was sampled.
print("Double+Dueling action:", a_next_w)
print("target:", round(float(rainbow_y_w), 3), "TD error:", round(float(td_rainbow_w), 3))
print("PER-weighted squared loss:", round(float(weighted_loss_w), 3))
assert a_next_w == 1 and round(float(rainbow_y_w), 3) == 1.690

▶ What you'll see: the chosen action comes from online dueling Q, the value from target dueling Q, and the loss is scaled by a replay weight.

In [ ]:
plt.figure(figsize=(5, 3))
plt.bar(["current Q", "Rainbow target", "TD error"], [q_sa_w, rainbow_y_w, td_rainbow_w], color=["gray", "seagreen", "crimson"])
plt.title("5: stacked variants still train a Bellman target")
plt.ylabel("value")
plt.show()

▶ What you'll see: Rainbow-style machinery changes how the target and loss are computed, but the update still reduces TD error.

*Why it's done this way:* DQN has several separable failure modes — overestimated max targets, weak action-gap representation, and inefficient replay. Combining fixes works because each one modifies a different mathematical object: selection/evaluation, value parameterization, and sample weighting.

## 🛠️ Setup

In [ ]:
import numpy as np # load NumPy for arrays, Bellman targets, sampling probabilities, and small numerical checks.
import matplotlib.pyplot as plt # load Matplotlib for compact plots that make Q-values, losses, and priorities inspectable.
np.random.seed(0) # make all random examples reproducible across notebook runs.

def dqn_target(r, gamma, q_next): # compute the standard DQN one-step target.
    return float(r + gamma * np.max(q_next)) # bootstrap from the largest next-action value.

def double_dqn_target(r, gamma, q_online_next, q_target_next): # compute Double DQN's choose/evaluate target.
    a = int(np.argmax(q_online_next)) # choose the next action with the online network.
    return float(r + gamma * q_target_next[a]), a # evaluate that chosen action with the target network.

def dueling_q(V, A): # combine a scalar/column value stream with an advantage matrix.
    A = np.asarray(A, dtype=float) # ensure advantages are numeric arrays.
    V = np.asarray(V, dtype=float) # ensure values broadcast cleanly.
    return V[..., None] + A - A.mean(axis=-1, keepdims=True) # subtract mean advantage for identifiability.

def per_probs(td_errors, alpha=0.6, eps=1e-3): # turn TD errors into prioritized replay probabilities.
    p = (np.abs(td_errors) + eps) ** alpha # convert surprise into positive priorities.
    return p / p.sum() # normalize priorities into a probability distribution.

def importance_weights(probs, beta=0.4): # compute normalized PER importance-sampling weights.
    n = len(probs) # number of replay items.
    w = (n * probs) ** (-beta) # inverse-probability correction.
    return w / w.max() # normalize for stable loss scaling.

## 🟢 Basics (warm-up)

### Basic 1 — Discount a short reward stream

**Goal.** Compute a return from rewards, because DQN targets are built from discounted future consequence. We build it in 2 steps.

In [ ]:
rewards_b1 = np.array([1.0, 0.0, 2.0]) # store immediate and delayed rewards for a tiny trajectory.
gamma_b1 = 0.9 # set the discount factor used throughout the lesson.
print("rewards:", rewards_b1) # inspect the reward stream before discounting.

In [ ]:
weights_b1 = gamma_b1 ** np.arange(len(rewards_b1)) # compute [1, gamma, gamma^2].
G_b1 = float(np.sum(weights_b1 * rewards_b1)) # compute discounted return.
print("discount weights:", np.round(weights_b1, 3), "return:", round(G_b1, 3)) # inspect the weighted sum.
assert round(G_b1, 3) == 2.620 # verify the canonical delayed-reward calculation.
plt.figure(figsize=(4, 3)) # create a compact contribution chart.
plt.bar(["r0", "γr1", "γ²r2"], weights_b1 * rewards_b1, color="teal") # show each discounted contribution.
plt.title("Basic 1: discounted return pieces") # title the plot.
plt.ylabel("contribution") # label the contribution scale.
plt.show() # display the chart.

▶ What you'll see: only the first and third rewards contribute, with the third reduced by γ².

👀 Takeaway: return is delayed consequence on a discounted scale, not just immediate reward.

### Basic 2 — Build one DQN target

**Goal.** Compute $r+\gamma\max Q(s',a')$, because DQN learns from one-step bootstrapped targets. We build it in 2 steps.

In [ ]:
q_next_b2 = np.array([0.8, 0.3, 0.6]) # define target-network values for three next actions.
r_b2 = 1.0 # define the observed immediate reward.
gamma_b2 = 0.9 # define the discount factor.
print("next Q values:", q_next_b2) # inspect candidate future values.

In [ ]:
y_b2 = dqn_target(r_b2, gamma_b2, q_next_b2) # compute the standard max-backup target.
print("DQN target:", round(y_b2, 3)) # inspect the target value.
assert round(y_b2, 3) == 1.720 # verify 1 + 0.9*0.8.
plt.figure(figsize=(4, 3)) # create a bar plot of next-action values.
plt.bar(["a0", "a1", "a2"], q_next_b2, color="steelblue") # draw each next action value.
plt.axhline(np.max(q_next_b2), color="black", linestyle="--") # mark the max used by the target.
plt.title("Basic 2: max next-action value") # title the plot.
plt.show() # display the chart.

▶ What you'll see: the largest next value is selected as the future term in the target.

👀 Takeaway: DQN bootstraps from the greedy next-action value.

### Basic 3 — Move an estimate toward a target

**Goal.** Apply a small value update, because bootstrapped targets should usually be approached gradually. We build it in 2 steps.

In [ ]:
q_old_b3 = 0.4 # current estimate for Q(s,a).
y_b3 = 1.72 # target computed from reward plus discounted future estimate.
alpha_b3 = 0.5 # learning rate for the visible scalar update.
print("old Q:", q_old_b3, "target:", y_b3) # inspect endpoints before the update.

In [ ]:
q_new_b3 = q_old_b3 + alpha_b3 * (y_b3 - q_old_b3) # move halfway toward the target.
print("new Q:", round(q_new_b3, 3)) # inspect the updated estimate.
assert round(q_new_b3, 3) == 1.060 # verify the worked update.
plt.figure(figsize=(4, 3)) # create a before-target-after chart.
plt.bar(["old", "target", "new"], [q_old_b3, y_b3, q_new_b3], color=["gray", "black", "green"]) # compare values.
plt.title("Basic 3: partial TD update") # title the plot.
plt.ylabel("Q value") # label the value scale.
plt.show() # display the chart.

▶ What you'll see: the new estimate lands midway between the old estimate and the target.

👀 Takeaway: the learning rate controls how strongly one transition changes Q.

### Basic 4 — Choose a greedy action

**Goal.** Use argmax over Q-values, because DQN's policy picks the action with highest estimated consequence. We build it in 2 steps.

In [ ]:
q_values_b4 = np.array([1.2, 0.9, 1.5, 0.4]) # define action values for one state.
actions_b4 = np.array(["left", "right", "up", "down"]) # define readable action labels.
print("Q values:", q_values_b4) # inspect the action-value row.

In [ ]:
greedy_idx_b4 = int(np.argmax(q_values_b4)) # choose the index of the largest value.
greedy_action_b4 = actions_b4[greedy_idx_b4] # map the index to an action label.
print("greedy action:", greedy_action_b4, "value:", q_values_b4[greedy_idx_b4]) # inspect the chosen action.
assert greedy_action_b4 == "up" # verify the highest value is action up.
plt.figure(figsize=(4, 3)) # create an action-value chart.
plt.bar(actions_b4, q_values_b4, color="orange") # draw one bar per action.
plt.title("Basic 4: greedy action from Q") # title the plot.
plt.ylabel("Q(s,a)") # label the Q-value scale.
plt.show() # display the chart.

▶ What you'll see: the tallest bar is selected by argmax.

👀 Takeaway: values become behavior through greedy or near-greedy action selection.

### Basic 5 — See max overestimation from noisy values

**Goal.** Compare individual estimate noise with max noise, because Double DQN exists to reduce max-selection bias. We build it in 2 steps.

In [ ]:
true_q_b5 = np.array([1.0, 1.0, 1.0]) # assume all actions are equally valuable in truth.
noise_b5 = np.array([0.2, -0.1, 0.4]) # define one noisy estimation error per action.
est_q_b5 = true_q_b5 + noise_b5 # create noisy estimated action values.
print("estimated Q:", est_q_b5) # inspect noisy values before taking a max.

In [ ]:
max_est_b5 = float(np.max(est_q_b5)) # max over noisy estimates.
mean_est_b5 = float(np.mean(est_q_b5)) # average estimate for comparison.
print("mean estimate:", round(mean_est_b5, 3), "max estimate:", round(max_est_b5, 3)) # inspect upward selection bias.
assert round(max_est_b5, 3) == 1.400 # verify the largest noisy value.
plt.figure(figsize=(4, 3)) # create a noise chart.
plt.bar(["a0", "a1", "a2"], est_q_b5, color="crimson") # show noisy estimates.
plt.axhline(1.0, color="black", linestyle="--", label="true value") # mark true common value.
plt.title("Basic 5: max picks positive noise") # title the plot.
plt.legend() # show the true-value reference.
plt.show() # display the chart.

▶ What you'll see: the max estimate is above the true value even though errors include positives and negatives.

👀 Takeaway: maximizing noisy values tends to select overestimates.

### Basic 6 — Separate Double DQN selection and evaluation

**Goal.** Compute a Double DQN target, because the online network should choose while the target network evaluates. We build it in 2 steps.

In [ ]:
online_next_b6 = np.array([1.2, 1.1, 0.7]) # online next-state values used for action selection.
target_next_b6 = np.array([0.55, 0.95, 0.65]) # target next-state values used for evaluation.
print("online:", online_next_b6, "target:", target_next_b6) # inspect both value rows.

In [ ]:
y_b6, a_b6 = double_dqn_target(1.0, 0.9, online_next_b6, target_next_b6) # compute Double DQN target and selected action.
print("selected action:", a_b6, "Double target:", round(y_b6, 3)) # inspect the separated backup.
assert a_b6 == 0 and round(y_b6, 3) == 1.495 # verify online selects a0 and target evaluates it.
plt.figure(figsize=(4, 3)) # create a comparison chart.
plt.bar(["plain max target", "Double target"], [1.0 + 0.9*np.max(target_next_b6), y_b6], color=["gray", "green"]) # compare targets.
plt.title("Basic 6: Double DQN target") # title the plot.
plt.ylabel("target value") # label the target scale.
plt.show() # display the chart.

▶ What you'll see: the Double target can be lower than the plain max target when the two networks disagree.

👀 Takeaway: Double DQN reduces overoptimistic targets by decoupling choice from evaluation.

### Basic 7 — Center advantages for dueling Q-values

**Goal.** Subtract the mean advantage, because dueling DQN needs an identifiable value/advantage split. We build it in 2 steps.

In [ ]:
V_b7 = np.array([2.0]) # define one state-value estimate.
A_b7 = np.array([[0.5, -0.2, 0.1]]) # define three raw action advantages.
print("V:", V_b7, "raw A:", A_b7) # inspect the two streams.

In [ ]:
Q_b7 = dueling_q(V_b7, A_b7) # combine value and centered advantages.
print("dueling Q:", np.round(Q_b7, 3)) # inspect action values.
assert round(float(np.mean(Q_b7)), 3) == 2.000 # verify the average Q equals V.
plt.figure(figsize=(4, 3)) # create a dueling Q chart.
plt.bar(["a0", "a1", "a2"], Q_b7[0], color="teal") # plot action values.
plt.axhline(V_b7[0], color="black", linestyle="--", label="V") # show the shared state value.
plt.title("Basic 7: V plus centered A") # title the plot.
plt.legend() # show the V reference.
plt.show() # display the chart.

▶ What you'll see: action values vary around the shared state value.

👀 Takeaway: dueling DQN learns what the state is worth and how actions differ from that baseline.

### Basic 8 — Convert TD errors to replay probabilities

**Goal.** Build prioritized replay probabilities, because surprising transitions should be replayed more often. We build it in 2 steps.

In [ ]:
td_b8 = np.array([0.05, 0.20, 1.00, 0.40, 2.00]) # define absolute TD errors from five replay items.
alpha_b8 = 0.6 # choose a moderate prioritization strength.
print("TD errors:", td_b8) # inspect surprise values.

In [ ]:
probs_b8 = per_probs(td_b8, alpha=alpha_b8) # convert TD errors into sampling probabilities.
print("probabilities:", np.round(probs_b8, 3)) # inspect the replay distribution.
assert int(np.argmax(probs_b8)) == 4 # verify the largest TD error gets largest probability.
plt.figure(figsize=(4, 3)) # create a replay-probability chart.
plt.bar(range(len(td_b8)), probs_b8, color="purple") # plot one probability per transition.
plt.title("Basic 8: prioritized replay probabilities") # title the plot.
plt.xlabel("transition") # label replay ids.
plt.ylabel("probability") # label probability scale.
plt.show() # display the chart.

▶ What you'll see: larger TD errors produce larger replay probabilities.

👀 Takeaway: PER changes what the learner sees most often by prioritizing surprise.

### Basic 9 — Compute PER importance weights

**Goal.** Correct non-uniform replay sampling, because prioritized samples would otherwise bias the loss. We build it in 2 steps.

In [ ]:
probs_b9 = per_probs(np.array([0.05, 0.20, 1.00, 0.40, 2.00]), alpha=0.6) # reuse replay probabilities.
beta_b9 = 0.4 # choose partial importance-sampling correction.
print("sampling probs:", np.round(probs_b9, 3)) # inspect the biased sampling distribution.

In [ ]:
weights_b9 = importance_weights(probs_b9, beta=beta_b9) # compute normalized correction weights.
print("importance weights:", np.round(weights_b9, 3)) # inspect loss weights.
assert round(float(np.max(weights_b9)), 3) == 1.000 # verify normalization.
plt.figure(figsize=(4, 3)) # create a weight chart.
plt.bar(range(len(weights_b9)), weights_b9, color="orange") # plot one correction per transition.
plt.title("Basic 9: PER importance weights") # title the plot.
plt.xlabel("transition") # label replay ids.
plt.ylabel("normalized weight") # label correction scale.
plt.show() # display the chart.

▶ What you'll see: rare transitions receive larger correction weights than frequently sampled ones.

👀 Takeaway: PER samples for learning speed but weights losses to reduce distribution bias.

### Basic 10 — Compute a TD-error loss

**Goal.** Turn a target and prediction into squared TD error, because DQN variants ultimately optimize prediction error. We build it in 2 steps.

In [ ]:
q_pred_b10 = 0.6 # current prediction for Q(s,a).
y_target_b10 = 1.81 # bootstrapped target from a variant backup.
weight_b10 = 0.8 # example replay importance weight.
td_b10 = y_target_b10 - q_pred_b10 # compute signed TD error.
print("TD error:", round(td_b10, 3)) # inspect the residual.

In [ ]:
loss_b10 = weight_b10 * td_b10 ** 2 # compute a weighted squared TD loss.
print("weighted loss:", round(loss_b10, 3)) # inspect scalar training objective.
assert round(loss_b10, 3) == 1.171 # verify the calculation.
plt.figure(figsize=(4, 3)) # create a loss ingredient chart.
plt.bar(["prediction", "target", "weighted loss"], [q_pred_b10, y_target_b10, loss_b10], color=["gray", "green", "red"]) # compare quantities.
plt.title("Basic 10: weighted TD loss") # title the plot.
plt.show() # display the chart.

▶ What you'll see: the loss grows with the squared gap between prediction and target.

👀 Takeaway: all these variants still train by shrinking TD error.

## 🟡 Easy

### Easy 1 — Compare DQN and Double DQN targets

**Goal.** Compute both targets on the same transition, because the difference isolates max-overestimation control. We build it in 3 steps.

In [ ]:
r_e1 = 1.0 # immediate reward for the sampled transition.
gamma_e1 = 0.9 # discount factor.
online_e1 = np.array([1.2, 1.1, 0.7]) # online next-state Q values.
target_e1 = np.array([0.55, 0.95, 0.65]) # target next-state Q values.
print("online:", online_e1, "target:", target_e1) # inspect the two estimators.

In [ ]:
dqn_e1 = dqn_target(r_e1, gamma_e1, target_e1) # plain DQN uses the target network's max value.
double_e1, chosen_e1 = double_dqn_target(r_e1, gamma_e1, online_e1, target_e1) # Double DQN separates choice/evaluation.
print("plain target:", round(dqn_e1, 3), "Double target:", round(double_e1, 3), "chosen:", chosen_e1) # inspect both backups.
assert round(dqn_e1, 3) == 1.855 and round(double_e1, 3) == 1.495 # verify the target difference.

In [ ]:
plt.figure(figsize=(4, 3)) # create a direct target comparison.
plt.bar(["DQN", "Double DQN"], [dqn_e1, double_e1], color=["crimson", "seagreen"]) # plot target values.
plt.title("Easy 1: target comparison") # title the plot.
plt.ylabel("backup target") # label target scale.
plt.show() # display the chart.

▶ What you'll see: the Double DQN target is smaller when online and target networks disagree about the best next action.

👀 Takeaway: Double DQN changes the target, not the meaning of Q.

### Easy 2 — Build a dueling Q table for two states

**Goal.** Combine value and advantage streams for multiple states, because dueling DQN produces a full action-value table. We build it in 3 steps.

In [ ]:
V_e2 = np.array([2.0, 0.5]) # two state values.
A_e2 = np.array([[0.5, -0.2, 0.1], [0.0, 0.3, -0.3]]) # raw advantages for three actions in each state.
print("V shape:", V_e2.shape, "A shape:", A_e2.shape) # inspect stream shapes.

In [ ]:
Q_e2 = dueling_q(V_e2, A_e2) # combine streams with mean-centered advantages.
print("Q table:\n", np.round(Q_e2, 3)) # inspect full action values.
print("row means:", np.round(Q_e2.mean(axis=1), 3)) # verify each row averages back to V.
assert np.allclose(np.round(Q_e2.mean(axis=1), 3), V_e2) # verify identifiability behavior.

In [ ]:
plt.figure(figsize=(4, 3)) # create a heatmap for the Q table.
plt.imshow(Q_e2, cmap="viridis", aspect="auto") # draw action values by state and action.
plt.colorbar(label="Q") # add a color scale.
plt.title("Easy 2: dueling Q table") # title the heatmap.
plt.xlabel("action") # label columns.
plt.ylabel("state") # label rows.
plt.show() # display the heatmap.

▶ What you'll see: each state's action values fluctuate around that state's value baseline.

👀 Takeaway: dueling parameterization shares a state baseline while preserving action preferences.

### Easy 3 — Sample a replay batch by priority

**Goal.** Draw replay indices from PER probabilities, because implementation requires turning priorities into actual minibatches. We build it in 3 steps.

In [ ]:
td_e3 = np.array([0.05, 0.20, 1.00, 0.40, 2.00]) # define replay TD errors.
probs_e3 = per_probs(td_e3, alpha=0.6) # compute replay probabilities.
rng_e3 = np.random.default_rng(3) # create a reproducible local sampler.
print("probabilities:", np.round(probs_e3, 3)) # inspect the categorical distribution.

In [ ]:
batch_e3 = rng_e3.choice(len(td_e3), size=12, replace=True, p=probs_e3) # sample replay ids according to PER.
counts_e3 = np.bincount(batch_e3, minlength=len(td_e3)) # count sampled ids.
print("sampled batch:", batch_e3) # inspect sampled transition ids.
print("counts:", counts_e3) # inspect empirical frequencies.
assert counts_e3.sum() == 12 # verify batch size.

In [ ]:
plt.figure(figsize=(4, 3)) # create a sampled-count chart.
plt.bar(range(len(td_e3)), counts_e3, color="purple") # plot how often each transition appeared.
plt.title("Easy 3: prioritized sample counts") # title the plot.
plt.xlabel("transition") # label replay ids.
plt.ylabel("count in batch") # label counts.
plt.show() # display the chart.

▶ What you'll see: high-priority ids usually appear more often, though a small batch is still random.

👀 Takeaway: prioritized probabilities become minibatches through categorical sampling.

### Easy 4 — Update priorities after learning

**Goal.** Replace old replay priorities with new TD errors, because PER priorities should track the model's current mistakes. We build it in 3 steps.

In [ ]:
old_td_e4 = np.array([0.05, 0.20, 1.00, 0.40, 2.00]) # TD errors before a learning step.
new_td_e4 = np.array([0.04, 0.80, 0.30, 0.35, 1.20]) # TD errors after recomputing selected transitions.
old_probs_e4 = per_probs(old_td_e4) # old sampling distribution.
new_probs_e4 = per_probs(new_td_e4) # updated sampling distribution.
print("old probs:", np.round(old_probs_e4, 3)) # inspect old priorities.

In [ ]:
print("new probs:", np.round(new_probs_e4, 3)) # inspect updated priorities.
changed_e4 = new_probs_e4 - old_probs_e4 # compute probability shifts.
print("probability shift:", np.round(changed_e4, 3)) # inspect which transitions gained priority.
assert int(np.argmax(new_probs_e4)) == 4 # verify transition 4 remains largest after update.

In [ ]:
x_e4 = np.arange(len(old_td_e4)) # x positions for grouped bars.
plt.figure(figsize=(5, 3)) # create a before-after priority chart.
plt.bar(x_e4 - 0.18, old_probs_e4, width=0.36, label="old", color="gray") # plot old probabilities.
plt.bar(x_e4 + 0.18, new_probs_e4, width=0.36, label="new", color="orange") # plot updated probabilities.
plt.title("Easy 4: priorities follow current TD error") # title the chart.
plt.xlabel("transition") # label transition ids.
plt.legend() # show labels.
plt.show() # display the chart.

▶ What you'll see: transitions whose TD errors rise gain sampling probability.

👀 Takeaway: PER is dynamic; priorities should be refreshed as the learner changes.

### Easy 5 — One tiny Double-Dueling loss

**Goal.** Combine Double DQN selection with dueling values and a PER weight, because real variants often stack cleanly. We build it in 3 steps.

In [ ]:
V_online_e5 = np.array([1.1]) # online value stream for one next state.
A_online_e5 = np.array([[0.2, 0.4, -0.1]]) # online advantages for action selection.
V_target_e5 = np.array([0.9]) # target value stream for evaluation.
A_target_e5 = np.array([[0.1, 0.0, 0.3]]) # target advantages for evaluation.
print("streams ready") # confirm ingredients.

In [ ]:
Q_online_e5 = dueling_q(V_online_e5, A_online_e5) # online dueling Q values.
Q_target_e5 = dueling_q(V_target_e5, A_target_e5) # target dueling Q values.
y_e5, action_e5 = double_dqn_target(1.0, 0.9, Q_online_e5[0], Q_target_e5[0]) # choose online, evaluate target.
print("online Q:", np.round(Q_online_e5, 3), "target Q:", np.round(Q_target_e5, 3)) # inspect both Q rows.
print("action:", action_e5, "target:", round(y_e5, 3)) # inspect the backup.
assert action_e5 == 1 and round(y_e5, 3) == 1.690 # verify the stacked target.

In [ ]:
q_pred_e5 = 0.6 # current Q(s,a) prediction.
per_weight_e5 = 0.75 # example importance weight from replay.
loss_e5 = per_weight_e5 * (y_e5 - q_pred_e5) ** 2 # compute weighted TD loss.
print("weighted loss:", round(loss_e5, 3)) # inspect scalar loss.
plt.figure(figsize=(4, 3)) # create a compact loss chart.
plt.bar(["prediction", "target", "loss"], [q_pred_e5, y_e5, loss_e5], color=["gray", "green", "red"]) # compare training quantities.
plt.title("Easy 5: Double-Dueling PER loss") # title the plot.
plt.show() # display the chart.

▶ What you'll see: several variants feed one final target and weighted loss.

👀 Takeaway: DQN variants are modular repairs to target construction, representation, and replay.

## 🔴 Advanced

### Advanced 1 — Simulate overestimation bias across many noisy actions

**Goal.** Estimate max bias empirically, because Double DQN's motivation is statistical rather than cosmetic. We build it in 4 steps.

In [ ]:
rng_a1 = np.random.default_rng(11) # create reproducible randomness.
true_q_a1 = 1.0 # all actions have the same true value.
n_trials_a1 = 4000 # number of noisy states to simulate.
actions_a1 = 8 # number of actions in each state.
print("trials:", n_trials_a1, "actions:", actions_a1) # inspect simulation size.

In [ ]:
noise_a1 = rng_a1.normal(0, 0.5, size=(n_trials_a1, actions_a1)) # draw independent estimate noise.
estimates_a1 = true_q_a1 + noise_a1 # create noisy Q estimates.
max_est_a1 = estimates_a1.max(axis=1) # plain max-selected estimates.
mean_max_a1 = float(np.mean(max_est_a1)) # average max estimate.
print("mean max estimate:", round(mean_max_a1, 3)) # inspect overestimation.
assert mean_max_a1 > true_q_a1 # verify positive max bias.

In [ ]:
noise_eval_a1 = rng_a1.normal(0, 0.5, size=(n_trials_a1, actions_a1)) # independent evaluator noise.
eval_estimates_a1 = true_q_a1 + noise_eval_a1 # target/evaluator estimates.
chosen_a1 = np.argmax(estimates_a1, axis=1) # actions selected by online estimates.
double_eval_a1 = eval_estimates_a1[np.arange(n_trials_a1), chosen_a1] # independent evaluation of chosen actions.
mean_double_a1 = float(np.mean(double_eval_a1)) # average Double-style estimate.
print("mean Double-style estimate:", round(mean_double_a1, 3)) # inspect reduced bias.
assert abs(mean_double_a1 - true_q_a1) < abs(mean_max_a1 - true_q_a1) # verify bias reduction in this simulation.

In [ ]:
plt.figure(figsize=(5, 3)) # create a bias comparison chart.
plt.hist(max_est_a1, bins=35, alpha=0.6, label="plain max", color="crimson") # show max estimate distribution.
plt.hist(double_eval_a1, bins=35, alpha=0.6, label="Double eval", color="seagreen") # show independent-eval distribution.
plt.axvline(true_q_a1, color="black", linestyle="--", label="true") # mark true value.
plt.title("Advanced 1: max overestimation bias") # title the plot.
plt.legend() # show labels.
plt.show() # display the histogram.

▶ What you'll see: the plain max distribution shifts right of the true value more than the Double-style evaluation.

👀 Takeaway: Double DQN reduces overestimation by not evaluating an action with the same noise that selected it.

### Advanced 2 — Train a tiny linear dueling head

**Goal.** Fit a NumPy-only dueling head to target Q-values, because dueling architecture is just a structured parameterization before any deep framework is involved. We build it in 4 steps.

In [ ]:
X_a2 = np.array([[1.0, 0.0], [0.0, 1.0], [1.0, 1.0]]) # three simple state feature vectors.
Y_a2 = np.array([[2.3, 1.6, 2.1], [0.4, 0.9, 0.1], [1.6, 1.8, 1.2]]) # target Q rows for three actions.
rng_a2 = np.random.default_rng(12) # reproducible initialization.
wV_a2 = 0.1 * rng_a2.normal(size=2) # linear value weights.
wA_a2 = 0.1 * rng_a2.normal(size=(2, 3)) # linear advantage weights.
print("X shape:", X_a2.shape, "Y shape:", Y_a2.shape) # inspect training shapes.

In [ ]:
losses_a2 = [] # store MSE over training iterations.
for step_a2 in range(500): # run simple full-batch gradient descent.
    V_a2 = X_a2 @ wV_a2 # value stream predictions.
    A_a2 = X_a2 @ wA_a2 # advantage stream predictions.
    Q_a2 = dueling_q(V_a2, A_a2) # dueling action values.
    err_a2 = Q_a2 - Y_a2 # residuals against target Q table.
    losses_a2.append(float(np.mean(err_a2 ** 2))) # record MSE.
    dQ_a2 = 2 * err_a2 / err_a2.size # gradient of mean squared error w.r.t. Q.
    dV_a2 = np.sum(dQ_a2, axis=1) # each action receives V equally.
    dA_center_a2 = dQ_a2 - dQ_a2.mean(axis=1, keepdims=True) # gradient through A - mean(A).
    wV_a2 -= 0.08 * (X_a2.T @ dV_a2) # update value weights.
    wA_a2 -= 0.08 * (X_a2.T @ dA_center_a2) # update advantage weights.
print("loss start/end:", round(losses_a2[0], 3), round(losses_a2[-1], 3)) # inspect learning progress.
assert losses_a2[-1] < losses_a2[0] # verify training reduced error.

In [ ]:
V_fit_a2 = X_a2 @ wV_a2 # final value stream.
A_fit_a2 = X_a2 @ wA_a2 # final advantage stream.
Q_fit_a2 = dueling_q(V_fit_a2, A_fit_a2) # final fitted Q table.
print("fitted Q:\n", np.round(Q_fit_a2, 2)) # inspect learned predictions.
print("target Q:\n", Y_a2) # inspect targets.

In [ ]:
plt.figure(figsize=(5, 3)) # create a training curve plot.
plt.plot(losses_a2, color="teal") # plot MSE over gradient steps.
plt.title("Advanced 2: NumPy dueling-head training") # title the plot.
plt.xlabel("step") # label optimization steps.
plt.ylabel("MSE") # label loss.
plt.show() # display the curve.

▶ What you'll see: the loss decreases as the value and advantage weights learn a structured Q table.

👀 Takeaway: dueling DQN is a mathematical decomposition of Q-values, not a dependency on a deep-learning library.

### Advanced 3 — Sweep PER alpha and beta

**Goal.** Visualize prioritization and correction strengths, because PER has two knobs with opposite effects. We build it in 4 steps.

In [ ]:
td_a3 = np.array([0.05, 0.20, 1.00, 0.40, 2.00]) # fixed TD errors for the sweep.
alphas_a3 = np.array([0.0, 0.4, 0.8, 1.0]) # prioritization strengths.
beta_a3 = 0.6 # importance-correction strength used for the plotted weights.
print("alphas:", alphas_a3) # inspect sweep values.

In [ ]:
prob_rows_a3 = [] # store one probability row per alpha.
weight_rows_a3 = [] # store matching importance weights.
for alpha_a3 in alphas_a3: # sweep priority exponent.
    probs_a3 = per_probs(td_a3, alpha=alpha_a3) # compute sampling probabilities.
    weights_a3 = importance_weights(probs_a3, beta=beta_a3) # compute corrections.
    prob_rows_a3.append(probs_a3) # save probabilities.
    weight_rows_a3.append(weights_a3) # save weights.
prob_rows_a3 = np.array(prob_rows_a3) # convert to a matrix for plotting.
weight_rows_a3 = np.array(weight_rows_a3) # convert to a matrix for plotting.
print("prob rows:\n", np.round(prob_rows_a3, 3)) # inspect how alpha changes sampling.
assert np.allclose(prob_rows_a3[0], np.ones_like(td_a3) / len(td_a3)) # alpha=0 is uniform.

In [ ]:
print("weights at alpha=1:", np.round(weight_rows_a3[-1], 3)) # inspect correction when prioritization is strongest.
assert round(float(np.max(weight_rows_a3[-1])), 3) == 1.000 # verify normalized weights.

In [ ]:
plt.figure(figsize=(5, 3)) # create a heatmap for the probability sweep.
plt.imshow(prob_rows_a3, cmap="viridis", aspect="auto") # draw probabilities by alpha and transition.
plt.colorbar(label="sample probability") # add color scale.
plt.yticks(range(len(alphas_a3)), [f"α={a}" for a in alphas_a3]) # label alpha rows.
plt.xlabel("transition") # label transition columns.
plt.title("Advanced 3: PER α sharpens sampling") # title the heatmap.
plt.show() # display the heatmap.

▶ What you'll see: alpha 0 is uniform, while larger alpha concentrates probability on high-error transitions.

👀 Takeaway: α controls how aggressively PER chases surprise, while β controls how much the loss corrects that bias.

### Advanced 4 — Compare uniform replay and PER learning speed

**Goal.** Simulate scalar TD-error reduction under two replay rules, because PER is meant to spend updates where errors are large. We build it in 4 steps.

In [ ]:
initial_errors_a4 = np.array([0.1, 0.2, 1.5, 0.4, 2.5, 0.3]) # starting absolute TD errors.
steps_a4 = 60 # number of replay updates.
rng_a4 = np.random.default_rng(14) # reproducible sampling.
errors_uniform_a4 = initial_errors_a4.copy() # errors updated by uniform replay.
errors_per_a4 = initial_errors_a4.copy() # errors updated by PER.
print("initial mean error:", round(float(initial_errors_a4.mean()), 3)) # inspect starting error.

In [ ]:
curve_uniform_a4 = [] # store uniform mean errors.
curve_per_a4 = [] # store PER mean errors.
for step_a4 in range(steps_a4): # run replay updates.
    idx_u_a4 = rng_a4.integers(len(errors_uniform_a4)) # sample uniformly.
    probs_p_a4 = per_probs(errors_per_a4, alpha=0.7) # sample according to current errors.
    idx_p_a4 = rng_a4.choice(len(errors_per_a4), p=probs_p_a4) # prioritized sample.
    errors_uniform_a4[idx_u_a4] *= 0.82 # pretend one update reduces that transition's error.
    errors_per_a4[idx_p_a4] *= 0.82 # same update effect for PER.
    curve_uniform_a4.append(float(errors_uniform_a4.mean())) # record uniform progress.
    curve_per_a4.append(float(errors_per_a4.mean())) # record PER progress.
print("final uniform mean:", round(curve_uniform_a4[-1], 3), "final PER mean:", round(curve_per_a4[-1], 3)) # inspect final errors.
assert curve_per_a4[-1] < curve_uniform_a4[-1] # verify PER was faster in this toy run.

In [ ]:
print("uniform final errors:", np.round(errors_uniform_a4, 3)) # inspect remaining errors.
print("PER final errors:", np.round(errors_per_a4, 3)) # inspect remaining errors after prioritized updates.

In [ ]:
plt.figure(figsize=(5, 3)) # create a learning-speed comparison.
plt.plot(curve_uniform_a4, label="uniform replay", color="gray") # plot uniform mean error.
plt.plot(curve_per_a4, label="prioritized replay", color="purple") # plot PER mean error.
plt.title("Advanced 4: PER spends updates on large errors") # title the plot.
plt.xlabel("replay update") # label update axis.
plt.ylabel("mean absolute TD error") # label error scale.
plt.legend() # show labels.
plt.show() # display the plot.

▶ What you'll see: prioritized replay usually lowers mean error faster by revisiting the largest mistakes.

👀 Takeaway: PER is an efficiency trick: it changes update order, not the Bellman target itself.

### Advanced 5 — Assemble a tiny Rainbow-style replay update

**Goal.** Run one full NumPy update with Double selection, dueling values, and PER weighting, because Rainbow is best understood as stacked small decisions. We build it in 5 steps.

In [ ]:
features_a5 = np.array([[1.0, 0.0], [0.0, 1.0]]) # two next-state feature vectors.
wV_online_a5 = np.array([1.0, 0.2]) # online value weights.
wA_online_a5 = np.array([[0.1, 0.4, -0.2], [0.0, 0.2, 0.1]]) # online advantage weights.
wV_target_a5 = np.array([0.8, 0.3]) # target value weights.
wA_target_a5 = np.array([[0.2, 0.0, 0.1], [0.1, 0.3, -0.1]]) # target advantage weights.
print("feature shape:", features_a5.shape) # inspect feature matrix.

In [ ]:
next_state_a5 = 1 # choose the second next state.
q_online_a5 = dueling_q(features_a5 @ wV_online_a5, features_a5 @ wA_online_a5) # online dueling Q table.
q_target_a5 = dueling_q(features_a5 @ wV_target_a5, features_a5 @ wA_target_a5) # target dueling Q table.
chosen_a5 = int(np.argmax(q_online_a5[next_state_a5])) # Double DQN online action selection.
print("online next row:", np.round(q_online_a5[next_state_a5], 3)) # inspect selection values.
print("target next row:", np.round(q_target_a5[next_state_a5], 3)) # inspect evaluation values.
assert chosen_a5 == 1 # verify action 1 is online-greedy.

In [ ]:
reward_a5 = 1.0 # observed reward.
gamma_a5 = 0.9 # discount.
q_current_a5 = 0.5 # current Q(s,a) prediction before update.
y_a5 = reward_a5 + gamma_a5 * q_target_a5[next_state_a5, chosen_a5] # Double-dueling target.
td_a5 = y_a5 - q_current_a5 # TD error.
print("target:", round(float(y_a5), 3), "TD error:", round(float(td_a5), 3)) # inspect backup residual.

In [ ]:
replay_td_a5 = np.array([0.2, abs(td_a5), 1.4, 0.6]) # replay errors including this transition.
probs_a5 = per_probs(replay_td_a5, alpha=0.6) # compute replay probabilities.
weights_a5 = importance_weights(probs_a5, beta=0.5) # compute correction weights.
loss_a5 = weights_a5[1] * td_a5 ** 2 # weighted squared TD loss for this sampled transition.
print("probs:", np.round(probs_a5, 3)) # inspect replay distribution.
print("sample weight:", round(float(weights_a5[1]), 3), "loss:", round(float(loss_a5), 3)) # inspect weighted loss.
assert loss_a5 > 0 # verify the update has nonzero learning signal.

In [ ]:
plt.figure(figsize=(5, 3)) # create a final Rainbow-style diagnostic chart.
plt.bar(["Q(s,a)", "target", "TD", "weighted loss"], [q_current_a5, y_a5, td_a5, loss_a5], color=["gray", "green", "red", "purple"]) # show the update pieces.
plt.title("Advanced 5: one stacked DQN-variant update") # title the plot.
plt.ylabel("value") # label value scale.
plt.show() # display the chart.

▶ What you'll see: online dueling values choose the action, target dueling values evaluate it, and PER scales the resulting TD loss.

👀 Takeaway: Rainbow-style updates are understandable once each component's mathematical role is separated.

---

# Reference walkthrough — original compact notebook

The sections above build every idea from scratch with detailed steps and worked examples. Below is the original compact notebook for this lesson, kept as a concise reference and for its practice prompts.

DQN variants each repair a different failure mode in bootstrapped value learning.

Double DQN changes target selection, dueling changes the Q parameterization, and prioritized replay changes which errors are replayed. Together they address over-estimation, action-insensitive states, and sparse learning signals. Save a copy to Drive to edit.

In [ ]:

import math
from dataclasses import dataclass

import matplotlib.pyplot as plt
import numpy as np

SEED = 11712
rng = np.random.default_rng(SEED)
ACTIONS = np.array([[-1, 0], [0, 1], [1, 0], [0, -1]])
ACTION_NAMES = np.array(["↑", "→", "↓", "←"])


@dataclass
class GridMDP:
    name: str
    rows: int
    cols: int
    start: int
    goals: tuple
    pits: tuple
    walls: tuple
    step_cost: float
    slip: float
    wind: tuple
    gamma: float = 0.9

    @property
    def n_states(self):
        return self.rows * self.cols

    @property
    def n_actions(self):
        return 4

    def to_pos(self, state):
        return divmod(int(state), self.cols)

    def to_state(self, row, col):
        return int(row * self.cols + col)

    def is_terminal(self, state):
        return int(state) in self.goals or int(state) in self.pits

    def in_wall(self, row, col):
        return self.to_state(row, col) in self.walls

    def move(self, state, action):
        if self.is_terminal(state):
            return int(state)
        row, col = self.to_pos(state)
        delta = ACTIONS[int(action)]
        new_row = row + int(delta[0]) + int(self.wind[0])
        new_col = col + int(delta[1]) + int(self.wind[1])
        blocked = new_row < 0 or new_row >= self.rows
        blocked = blocked or new_col < 0 or new_col >= self.cols
        if blocked:
            return int(state)
        if self.in_wall(new_row, new_col):
            return int(state)
        return self.to_state(new_row, new_col)

    def reward(self, next_state):
        if int(next_state) in self.goals:
            return 1.0
        if int(next_state) in self.pits:
            return -1.0
        return float(self.step_cost)

    def transitions(self, state, action):
        if self.is_terminal(state):
            return [(1.0, int(state), 0.0)]
        intended = int(action)
        probs = {intended: 1.0 - self.slip}
        if self.slip > 0:
            probs[(intended - 1) % 4] = probs.get((intended - 1) % 4, 0.0) + self.slip / 2.0
            probs[(intended + 1) % 4] = probs.get((intended + 1) % 4, 0.0) + self.slip / 2.0
        out = []
        for actual, prob in probs.items():
            next_state = self.move(state, actual)
            out.append((prob, next_state, self.reward(next_state)))
        return out

    def step(self, state, action, local_rng):
        options = self.transitions(state, action)
        probs = np.array([item[0] for item in options], dtype=float)
        choice = int(local_rng.choice(len(options), p=probs))
        _, next_state, reward = options[choice]
        return next_state, reward, self.is_terminal(next_state)


def make_f12_ladder():
    return [
        GridMDP("D1 two-state chain", 1, 2, 0, (1,), (), (), -0.01, 0.0, (0, 0)),
        GridMDP("D2 slippery three-state chain", 1, 3, 0, (2,), (), (), -0.02, 0.2, (0, 0)),
        GridMDP("D3 4x4 gridworld", 4, 4, 0, (15,), (5,), (), -0.03, 0.05, (0, 0)),
        GridMDP("D4 windy stochastic grid", 4, 5, 0, (19,), (7, 13), (6,), -0.04, 0.15, (0, 0)),
        GridMDP("D5 sparse 6x6 grid", 6, 6, 0, (35,), (14, 21, 28), (8, 9, 16), -0.02, 0.1, (0, 0)),
    ]


def value_iteration(env, iterations=200, tol=1e-10):
    values = np.zeros(env.n_states)
    q_values = np.zeros((env.n_states, env.n_actions))
    for _ in range(iterations):
        old = values.copy()
        for state in range(env.n_states):
            for action in range(env.n_actions):
                total = 0.0
                for prob, next_state, reward in env.transitions(state, action):
                    total += prob * (reward + env.gamma * old[next_state])
                q_values[state, action] = total
            values[state] = np.max(q_values[state])
            if env.is_terminal(state):
                values[state] = 0.0
        if np.max(np.abs(values - old)) < tol:
            break
    for state in range(env.n_states):
        for action in range(env.n_actions):
            total = 0.0
            for prob, next_state, reward in env.transitions(state, action):
                total += prob * (reward + env.gamma * values[next_state])
            q_values[state, action] = total
    return values, q_values


def epsilon_greedy(q_values, state, epsilon, local_rng):
    if local_rng.random() < epsilon:
        return int(local_rng.integers(q_values.shape[1]))
    return int(np.argmax(q_values[state]))


def run_episode_return(env, q_values, epsilon, local_rng, max_steps=80):
    state = env.start
    total = 0.0
    discount = 1.0
    for _ in range(max_steps):
        action = epsilon_greedy(q_values, state, epsilon, local_rng)
        next_state, reward, done = env.step(state, action, local_rng)
        total += discount * reward
        discount *= env.gamma
        state = next_state
        if done:
            break
    return total


def state_features(env, state, mode="normalized"):
    row, col = env.to_pos(state)
    if mode == "one_hot":
        x = np.zeros(env.n_states)
        x[state] = 1.0
        return x
    if mode == "raw":
        return np.array([1.0, float(row), float(col), float(row * col)])
    row_scale = max(env.rows - 1, 1)
    col_scale = max(env.cols - 1, 1)
    return np.array([1.0, row / row_scale, col / col_scale, row * col / max(row_scale * col_scale, 1)])


def plot_value_and_curve(ladder, artifacts, curve, ylabel, title):
    fig, axes = plt.subplots(2, len(ladder), figsize=(3 * len(ladder), 5))
    for idx, env in enumerate(ladder):
        values = artifacts[idx].reshape(env.rows, env.cols)
        axes[0, idx].imshow(values, cmap="viridis")
        axes[0, idx].set_title(env.name)
        axes[0, idx].set_xticks([])
        axes[0, idx].set_yticks([])
        axes[1, idx].plot(curve[idx])
        axes[1, idx].set_xlabel("episode")
        axes[1, idx].set_ylabel(ylabel)
    fig.suptitle(title)
    plt.tight_layout()


def print_ladder_preview(ladder):
    rows = []
    for name_idx, env in enumerate(ladder, start=1):
        rows.append((f"D{name_idx}", env.name, env.n_states, env.n_actions, env.start, env.goals))
    for row in rows:
        print(row)
    sample = ladder[-1]
    print("D5 sample transition", sample.transitions(sample.start, 1))


## The concept, built once (D1)

The lesson keeps the Bellman language visible:

$$V^\pi(s)=\sum_a\pi(a\mid s)\sum_{s'}P(s'\mid s,a)\bigl(R(s,a,s')+\gamma V^\pi(s')\bigr)$$

The worked numbers used throughout Part 11 are $G=1+0.9\cdot0+0.9^2\cdot2=2.620$, $y=1+0.9\cdot0.8=1.720$, and $Q_{new}=0.4+0.5(1.720-0.4)=1.060$.


In [ ]:

reward = 1.0
gamma = 0.9
online_next = np.array([0.8, 0.7])
target_next = np.array([0.6, 1.2])
vanilla_target = reward + gamma * np.max(target_next)
double_action = int(np.argmax(online_next))
double_target = reward + gamma * target_next[double_action]
assert double_action == 0
assert np.isclose(vanilla_target, 2.080)
assert np.isclose(double_target, 1.540)
print(vanilla_target, double_target)


Now combine the Double target with a dueling decomposition and a prioritized replay sampling rule.

In [ ]:

def init_tiny_network(env, hidden=8, seed=0):
    local_rng = np.random.default_rng(seed)
    input_dim = env.n_states
    w1 = local_rng.normal(0.0, 0.08, size=(input_dim, hidden))
    b1 = np.zeros(hidden)
    w2 = local_rng.normal(0.0, 0.08, size=(hidden, env.n_actions))
    b2 = np.zeros(env.n_actions)
    return {"w1": w1, "b1": b1, "w2": w2, "b2": b2}


def one_hot(env, state):
    x = np.zeros(env.n_states)
    x[state] = 1.0
    return x


def dqn_forward(params, x):
    z1 = x @ params["w1"] + params["b1"]
    h1 = np.maximum(z1, 0.0)
    q_values = h1 @ params["w2"] + params["b2"]
    return z1, h1, q_values


def dqn_update(params, target_params, env, batch, alpha=0.03, gamma=0.9, double=False):
    for state, action, reward, next_state, done in batch:
        x = one_hot(env, state)
        z1, h1, q_values = dqn_forward(params, x)
        _, _, next_online = dqn_forward(params, one_hot(env, next_state))
        _, _, next_target = dqn_forward(target_params, one_hot(env, next_state))
        if double:
            next_action = int(np.argmax(next_online))
            bootstrap = next_target[next_action]
        else:
            bootstrap = np.max(next_target)
        target = reward + gamma * bootstrap * (not done)
        error = q_values[action] - target
        grad_q = np.zeros(env.n_actions)
        grad_q[action] = error
        grad_w2 = np.outer(h1, grad_q)
        grad_b2 = grad_q
        grad_h = params["w2"] @ grad_q
        grad_z = grad_h * (z1 > 0)
        grad_w1 = np.outer(x, grad_z)
        grad_b1 = grad_z
        params["w2"] -= alpha * grad_w2
        params["b2"] -= alpha * grad_b2
        params["w1"] -= alpha * grad_w1
        params["b1"] -= alpha * grad_b1
    return params


def tiny_dqn(env, episodes=60, epsilon=0.2, seed=0, use_replay=True, use_target=True, double=False):
    local_rng = np.random.default_rng(seed)
    params = init_tiny_network(env, seed=seed)
    target_params = {key: value.copy() for key, value in params.items()}
    replay = []
    returns = []
    for episode in range(episodes):
        state = env.start
        total = 0.0
        discount = 1.0
        for _ in range(60):
            _, _, q_values = dqn_forward(params, one_hot(env, state))
            if local_rng.random() < epsilon:
                action = int(local_rng.integers(env.n_actions))
            else:
                action = int(np.argmax(q_values))
            next_state, reward, done = env.step(state, action, local_rng)
            transition = (state, action, reward, next_state, done)
            replay.append(transition)
            if len(replay) > 100:
                replay.pop(0)
            if use_replay and len(replay) >= 8:
                idx = local_rng.choice(len(replay), size=8, replace=False)
                batch = [replay[int(i)] for i in idx]
            else:
                batch = [transition]
            active_target = target_params if use_target else params
            params = dqn_update(params, active_target, env, batch, double=double)
            total += discount * reward
            discount *= env.gamma
            state = next_state
            if done:
                break
        if use_target and episode % 10 == 0:
            target_params = {key: value.copy() for key, value in params.items()}
        returns.append(total)
    values = np.zeros(env.n_states)
    for state in range(env.n_states):
        _, _, q_values = dqn_forward(params, one_hot(env, state))
        values[state] = np.max(q_values)
    return params, values, returns

def dueling_q(value, advantages):
    return value + advantages - np.mean(advantages)


def prioritized_sample(priorities, fraction=0.2):
    count = max(1, int(math.ceil(len(priorities) * fraction)))
    order = np.argsort(priorities)[::-1]
    return order[:count]


def double_dueling_per_step(reward, gamma, online_next, target_next, value, advantages):
    action = int(np.argmax(online_next))
    double_target = reward + gamma * target_next[action]
    dueling = dueling_q(value, advantages)
    priorities = np.abs(np.array([0.2, -1.3, 0.4, 2.0, -0.1]))
    top = prioritized_sample(priorities, fraction=0.2)
    return action, double_target, dueling, top


## The dataset ladder
We build the F12 D1–D5 environment ladder inline: a two-state chain, a slippery chain, a 4x4 grid, a windy stochastic grid, and a larger sparse-reward grid.

In [ ]:

ladder = make_f12_ladder()
print_ladder_preview(ladder)


## Run the same method across D1–D5
Metric: return. The loops are tiny, seeded, and CPU-only.

In [ ]:

ladder = make_f12_ladder()
metrics = []
artifacts = []
curves = []
vanilla_curves = []
for idx, env in enumerate(ladder):
    _, vanilla_values, vanilla_returns = tiny_dqn(env, episodes=35, seed=SEED + idx, use_replay=True, use_target=True, double=False)
    _, values, returns = tiny_dqn(env, episodes=35, seed=SEED + idx, use_replay=True, use_target=True, double=True)
    metrics.append(np.mean(returns[-10:]))
    artifacts.append(values)
    curves.append(returns)
    vanilla_curves.append(vanilla_returns)
print("rung | metric=return")
for idx, value in enumerate(metrics, start=1):
    print(f"D{idx}: {value:.4f}")


## Results visualization
The closing figure shows value/policy heatmaps for each rung and a learning curve for the same metric.

In [ ]:

plot_value_and_curve(ladder, artifacts, curves, "return", "DQN variants: value heatmaps and return curves")
plt.figure(figsize=(6, 3))
plt.plot([np.mean(c[-10:]) for c in vanilla_curves], label="vanilla")
plt.plot([np.mean(c[-10:]) for c in curves], label="Double target")
plt.xticks(range(5), ["D1", "D2", "D3", "D4", "D5"])
plt.ylabel("late return")
plt.legend()
plt.title("Variant comparison across rungs")


### Pitfall on D5: over-estimation from a moving max
Vanilla max selection can evaluate the same noisy action it selected. Double DQN selects with the online network and evaluates with the target network.

In [ ]:

reward = 1.0
gamma = 0.9
online_next = np.array([0.8, 0.7])
target_next = np.array([0.6, 1.2])
vanilla_target = reward + gamma * np.max(target_next)
action, double_target, dueling, top = double_dueling_per_step(reward, gamma, online_next, target_next, 0.5, np.array([0.2, -0.2]))
print("vanilla max target", round(vanilla_target, 3))
print("Double DQN target", round(double_target, 3))
print("dueling Q values", np.round(dueling, 3))
print("top 20 percent replay index", top.tolist())


## Evaluate it + Practice
- Compare the metric against a no-skill random-action baseline before trusting the learned policy.
- Sanity check D1 first: the backed-up value should move toward the exact worked target.
- Ablate the key idea for this lesson and confirm the metric gets worse or less stable.
- Failure signals: exploding values, flat returns, unvisited actions, or mismatched table shapes.

Practice prompts:
1. Change the discount from 0.9 to 0.7 and predict which rungs lose the most value.


2. Add one extra pit to D5 and rerun the same metric table.

3. Replace the policy heatmap with arrows from `ACTION_NAMES[np.argmax(Q, axis=1)]` where Q is available.